## Stage 1: PlasticESM, MLM post-training of ESM-C on the PETadex corpus

Lead     : `Purav Gupta`

Start    : `2026-10-16`

Status   : single-seed run complete and verified; dual-seed run in flight

Files    : `src/petabite/model_module/lora.py`, `src/petabite/mlm_finetune.py`, `scripts/`

This is the lab notebook for Stage 1. Stage 2, the activity labels and linear
probing, lives in the `igem-toronto` repo and is deliberately out of scope here.

## What Stage 1 is

Continue pretraining ESM-C 300M on the PETadex corpus with a masked-LM
objective, training LoRA adapters only and keeping the backbone frozen. The
result, PlasticESM, is the backbone Stage 2 puts an activity head on.

The objective is masked amino acid prediction on **sequence alone**. No
metadata of any kind enters the loss: not HMM family, not signal peptide, not
ProtParam, not taxonomy. That is deliberate, and the reasoning is in the
"metadata" section below.

## The repo did not work when this started

Four independent failures, each verified by running `origin/main` as it stood:

| | |
|---|---|
| `AutoTokenizer.from_pretrained` | `ValueError: Tokenizer class EsmcTokenizer does not exist` |
| `inject_fused_lora` | `RuntimeError: found no fused modules to wrap (qkv=0, ffn=0)` |
| `dump_pip_freeze` | `FileNotFoundError: 'uv'` |
| `config/mlm.yaml` | `eval_split: validation`, a split the corpus does not have |

The second is the one that mattered. `fused_lora.py` dispatched on
`_PyTorchLayerNormLinear` and `_PyTorchLayerNormMLP`; `esm` 3.4.1 renamed those
to `EsmcLayerNormLinear` and `EsmcLayerNormMLP`, so the tree walk matched
nothing. Earlier versions injected zero adapters and trained only the 30 PEFT
`out_proj` adapters while reporting a healthy loss curve. main's guard did at
least refuse to continue, but it blamed `transformer-engine`, which is not
installed.

Separately, saving was broken. PEFT and a hand-rolled mechanism both ran, and
`get_peft_model_state_dict` keeps only keys containing `lora_` **and**
`.default.`. The hand-rolled parameters matched the first test and failed the
second, so a checkpoint held 1,843,200 of 14,745,600 parameters at r=32.

`model_module/lora.py` replaces both with one mechanism over all four
projections per block, owning its own save and load, with three guards: raise
if nothing is adapted, prove each wrapper is bit-identical to the module it
replaced at init, and compare the saved file against the live model.

In [ ]:
# Reproduce the parameter arithmetic the brief reported, which is also the
# check that the adapter layout is understood correctly.
from transformers import AutoModelForMaskedLM
from petabite.model_module.lora import LoraConfig, apply_lora

model = AutoModelForMaskedLM.from_pretrained("biohub/ESMC-300M", trust_remote_code=True)
report = apply_lora(model, LoraConfig(r=32, alpha=64))

out_proj_only = sum(p.numel() for n, p in model.named_parameters()
                    if p.requires_grad and "out_proj" in n)

print(report.summary())
print(f"out_proj alone      : {out_proj_only:,}   (what PEFT used to save)")
print(f"fraction that was lost: {1 - out_proj_only / report.trainable_params:.1%}")

## The corpus

The published corpus is the 18,172,960 90%-identity centroids. One epoch over
all of them is about 57 h on an L40S, and most of the extra data is
near-duplicates, so Stage 1 trains on one member per **60%-identity** cluster
instead: 1,814,296 clusters, about 5.5 h per epoch.

Nothing in the parquet marks which 90%-centroid represents its 60% cluster, so
a rule had to be chosen, and the choice is not cosmetic. Picking each cluster's
largest member gives a subset that is **89.7% Logan** with a median length of
306, because Logan dominates the corpus and its ORFs are called stop to stop.
`build_c60_corpus.py` draws one member per cluster uniformly at random instead,
seeded, which is unbiased with respect to origin and length.

**81 clusters are excluded** because they contain a sequence from the PETai
label set. The exclusion uses the full 1,219-sequence label set rather than one
split's val/test, so it stays valid however Stage 2 re-splits later. No
downstream evaluation sequence is seen during pretraining in any form.

In [ ]:
import json
from pathlib import Path

for seed in (0, 1):
    path = Path(f"/project/6122149/purav/petabite-tokenized/c60-seed{seed}/manifest.json")
    if not path.exists():
        continue
    m = json.loads(path.read_text())
    print(f"seed {m['seed']}: {m['sequences']:,} sequences, {m['tokens']:,} tokens, "
          f"mean {m['mean_tokens']}, {m['clusters_excluded']} clusters excluded")

## Throughput

Measured on one L40S, 15 timed steps after 4 warmup, projected over the corpus.
Tokens per second rather than steps per second, because every knob being tuned
changes how many tokens a step contains.

| | step | tokens/s | peak mem | projected epoch |
|---|---|---|---|---|
| baseline: batch 8 x 1024, fp32, no TF32, unfused AdamW, SAE on, padded to longest | 0.783 s | 10,467 | 23.5 GB | 14.83 h |
| tuned: batch 32 x 512, bf16, TF32, fused AdamW, SAE off | 0.562 s | 26,610 | 31.2 GB | 5.83 h |
| plus length bucketing at batch 48 | 0.524 s | 27,497 | 44.1 GB | 5.65 h |

**2.63x**, 14.8 h to 5.65 h. Most of it is bf16 with TF32 and fused AdamW.
Truncating at 512 rather than 1024 is nearly free in coverage, since the corpus
median is 306 residues. Length bucketing is worth only 3% and costs 13 GB, so
the config keeps batch 32: at this point the run is compute-bound, not
padding-bound.

`compute_sae` is **not** a speed lever. ESM-C defaults it on and it runs a
sparse autoencoder head masked-LM training never reads, so it is switched off
on principle, but measurement found no difference beyond noise.

The single biggest remaining win would be `transformer-engine`, which fuses
LayerNorm+Linear and LayerNorm+MLP. At 27.5k tokens/s this is near 9% MFU on a
362 TFLOPS card. It is not taken because it swaps the fused modules for TE's
classes, which `lora.py` does not wrap, so `apply_lora` would raise. That is
the guard working as intended.

## A 24-hour stall that was not a hang

The first production run sat at `0/9423` for 13 minutes with the GPU at 0%, the
main process pegged on one core and all 8 dataloader workers idle.

`datasets` 5.0.0 changed `dataset[column]` to return a lazy `Column` rather
than a list. transformers reads the lengths that way for `group_by_length`, and
`LengthGroupedSampler` then indexes the result once per comparison while
sorting each megabatch. One lazy access costs 3.4 ms against 0.2 us for a list,
and the sampler needs roughly 23.5M of them: about **23 hours** of building the
batch order, before step one, inside a 12 h job.

Nothing errors. `Column` supports `len()` and indexing, so the sampler accepts
it and is merely 16,700x slower, which presents as a run that looks slow to
start rather than one that is broken.

`read_lengths` reads the Arrow column directly, 0.018 s for the same 1.8M
values. Because that only works in storage order, `split_tokenized` draws its
own permutation and applies it to both the rows and the lengths, and
`read_lengths` refuses a dataset carrying an indices mapping rather than
returning a silently misaligned array.

In [ ]:
import time
from petabite.data_module.tokenized import load_tokenized, read_lengths

corpus = load_tokenized("/project/6122149/purav/petabite-tokenized/c60-seed0")
lazy_column = corpus["length"]
print("type of corpus['length']:", type(lazy_column).__name__)

start = time.time()
_ = [lazy_column[i] for i in range(2000)]
lazy_per_access = (time.time() - start) / 2000

start = time.time()
lengths = read_lengths(corpus)
read_seconds = time.time() - start

start = time.time()
_ = [lengths[i] for i in range(2000)]
list_per_access = (time.time() - start) / 2000

print(f"lazy access  : {lazy_per_access * 1e3:.3f} ms each")
print(f"array access : {list_per_access * 1e6:.3f} us each  "
      f"({lazy_per_access / list_per_access:,.0f}x faster)")
print(f"read_lengths : {read_seconds:.3f} s for {len(lengths):,} values")
print(f"23.5M lazy accesses would be {23.5e6 * lazy_per_access / 3600:.1f} h")

## Run 1: single seed, one epoch

Job 6318656, 1,809,215 sequences, 9,423 steps, **5 h 17 m**.
https://wandb.ai/morphius/petabite-esmc/runs/4dwq33oj

| | |
|---|---|
| train loss | 1.8545 -> **1.678** |
| eval loss | 1.7168 -> **1.6836** |
| adapters | 240 tensors, 14,745,600 parameters, 59 MB |
| verification | completeness, trained, reload: all pass |

Most of the gain is in the first 10% of the epoch (1.854 to 1.715), then it
grinds slowly to 1.678. Eval tracks train with no divergence, so nothing is
overfitting, but it does say a second pass over the same data would buy little.

The up-projections moved from 3.3e-4 after 20 steps to 5.2e-3 at the end, so
gradients reached all 120 adapters, which is what `verify_checkpoint.py`
checks: an untrained adapter round-trips perfectly and is still worthless.

## Two runs collided, and what survived

Jobs 6318367 and 6318656 ran the identical configuration for 5 h 17 m each,
on kn089 and kn026, and **both wrote `outputs/plasticesm-c60-r32`**. Their final
eval losses differ only at the 4th decimal (1.68877 against 1.68867), which is
GPU nondeterminism.

The final artefact is clean: it is timestamped inside the surviving run's
finish, after the other had exited, and that run wrote and verified it as one
atomic sequence. An independent re-verification passed afterwards. The
intermediate checkpoints belonged to whichever job reached them last and were
deleted.

Two guards now exist in `train_mlm.sbatch`: the job id goes in `OUTDIR` by
default, and the script refuses to start when another `plasticesm` job is
queued or running.

## Run 2: two independent draws rather than two epochs

Because the corpus builder draws one member per cluster at random, `--seed 1`
picks a **different** member of most clusters. Loading both draws costs the same
wall time as two epochs and shows the model more distinct protein.

How much more is a real question rather than a rhetorical one, and the answer
is less than it first appears: **48.7% of the 60% clusters are singletons**, so
both seeds are forced to pick the same member there.

| | sequences seen | distinct |
|---|---|---|
| 2 epochs over one draw | 3.63M | 1,814,296 |
| seed0 + seed1 | 3.63M | **2,494,836** (1.38x) |

So a second draw adds 680,540 genuinely new sequences, not 1.8M. Still strictly
better than a second epoch, which adds none, but 1.38x rather than 2x.

In [ ]:
import collections
import pyarrow.parquet as pq

parquet = pq.ParquetFile(
    "/project/6122149/purav/petadex-corpus/centroids90/centroids90.parquet"
)
members_per_cluster = collections.Counter()
for group in range(parquet.num_row_groups):
    for cluster in parquet.read_row_group(group, columns=["c60_id"])["c60_id"].to_pylist():
        members_per_cluster[cluster] += 1

histogram = collections.Counter(members_per_cluster.values())
total = len(members_per_cluster)
# A second independent draw lands on a new member with probability 1 - 1/k.
expected_new = sum(n * (1 - 1.0 / k) for k, n in histogram.items())

print(f"60% clusters : {total:,}")
print(f"  singletons : {histogram[1]:,} ({100 * histogram[1] / total:.1f}%)")
print(f"expected new from a 2nd draw: {expected_new:,.0f}")
print(f"  one draw   : {total:,} distinct")
print(f"  two draws  : {total + expected_new:,.0f} distinct ({(total + expected_new) / total:.2f}x)")

## Mistakes worth recording

Three, because each cost real time and each is now guarded against.

**A duplicate 5-hour run.** I relaunched without checking whether a production
run was already queued, so two identical jobs burned about 5.3 GPU-hours each.
`train_mlm.sbatch` now refuses to start alongside another `plasticesm` job.

**A corpus list that silently halved itself.** `sbatch --export` separates
variables with commas, so `--export=ALL,CORPUS="path0,path1"` arrives inside
the job as `CORPUS=path0`. Job 6334352 therefore trained one corpus while
reporting success; it was caught 3 minutes in only because the step count read
9,423 rather than the expected 18,873. The separator is now a colon, the
resolved list and its length are echoed, and a missing path exits before the
GPU is touched.

**A misdiagnosis of the stall.** The first hypothesis was that reading the
length column through a shuffle indices mapping was slow. Timing
`dataset["length"]` showed 0.00 s and appeared to clear it. That measured the
wrong half: the read is lazy, and the cost is in the 23.5M element accesses
afterwards.

## Why no metadata

ESM-C's corpus layers offer HMM family, signal peptide and ProtParam
descriptors at 100% coverage, and taxonomy and structure more sparsely. None of
the first three are used, for a reason worth stating precisely because it does
not generalise.

HMM family, SignalP and ProtParam are **cheap** functions of the sequence. A
300M model can plausibly learn them unaided, so supplying them adds an
inductive bias at best, not information.

Structure is an **expensive** function of the sequence: a 6B ESMFold2 trunk, 32
sampling steps, 3 loops. The 300M model cannot derive it at any amount of
training, so distilling it transfers computation the student cannot perform.
That asymmetry is why structure remains interesting and the rest does not.

Taxonomy was checked and rejected on coverage: **5.5%** on the 60% subset, not
the 54% an early biased sample suggested. The chain organism to BacDive to
growth temperature is therefore dead at the first link as well as the last.
`melting_point` and `thermostability` are null on all 307,155,746 corpus rows,
so there is no temperature target to distil from at all.

## Open

- The probe. Stock ESM-C against PlasticESM on the Stage 2 labels, which is the
  only measurement that says whether any of this helped. Nothing here claims a
  downstream gain until it runs.
- Structure distillation. About 1.79M ESMFold2 structures exist for this exact
  corpus, roughly 98.6% coverage, with per-structure pLDDT. Sizing the
  extraction is in progress; the objective itself is not built.
- Whether the dual-seed run beats the single-seed one, which is the direct test
  of whether more distinct sequence helps at this scale.